### POC: controlling precision and recall with `MultiClassificationLTTController`

This notebook is a proof of concept showing how to use
`MultiClassificationLTTController` to jointly control two performance metrics
on a synthetic multiclass classification problem: one-vs-rest **precision**
and **recall**, for every class.

We generate a synthetic dataset, fit a `LogisticRegression` classifier, then
calibrate a `MultiClassificationLTTController` with a single probability
threshold (applied to the predicted class' confidence). The controller finds,
among the candidate thresholds, those that statistically guarantee precision
and recall above their respective target levels for every class, at a chosen
confidence level.

In [2]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

from mapie.risk_control import MultiClassificationLTTController
from mapie.risk_control.risks import build_precision_ovr, build_recall_ovr
from mapie.utils import train_conformalize_test_split
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

#### Synthetic data and base classifier

We generate a synthetic dataset with `n_classes` classes and split it into
train (to fit the classifier), calibration (to calibrate the controller), and
test (to evaluate the final predictions) sets.

In [18]:
n_classes = 5

X, y = make_classification(
    n_samples=4000,
    n_features=10,
    n_informative=10,
    n_redundant=0,
    n_classes=n_classes,
    n_clusters_per_class=1,
    random_state=RANDOM_STATE,
)

clf = LogisticRegression(max_iter = 1000).fit(X, y)
class_labels = np.unique(y)

/Users/trezel/Documents/code/MAPIE/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/trezel/Documents/code/MAPIE/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/trezel/Documents/code/MAPIE/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning:

invalid value encountered in matmul

/Users/trezel/Documents/code/MAPIE/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning:

divide by zero encountered in matmul

/Users/trezel/Documents/code/MAPIE/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning:

overflow encountered in matmul

/Users/trezel/Documents/code/MAPIE/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning:

invalid value encountered in matmul



#### Risks: one-vs-rest precision and recall per class

`build_precision_ovr` and `build_recall_ovr` each build one `ClassSpecificRisk`
per class label. We control both metrics for every class at once by
concatenating the two lists of risks, with one target level per risk: here,
a precision target of 0.7 and a recall target of 0.7, for each class.

We pick one of the recall risks as the secondary objective
(`best_predict_param_choice`): among the thresholds that jointly satisfy every
precision and recall constraint, the controller selects the one minimizing
that secondary risk (i.e. maximizing recall for the first class).

In [19]:
target_precision = 0.7
target_recall = 0.7

precision_risks = build_precision_ovr(class_labels)
recall_risks = build_recall_ovr(class_labels)
risks = precision_risks + recall_risks
target_levels = [target_precision] * len(precision_risks) + [target_recall] * len(
    recall_risks
)

controller = MultiClassificationLTTController(
    predict_function=clf.predict_proba,
    risks=risks,
    target_level=target_levels,
    confidence_level=0.8,
    best_predict_param_choice=precision_risks[0],
    list_predict_params=np.linspace(0.05, 0.95, 20),
    fwer_method="bonferroni_holm",
)
controller.calibrate(X, y)

print(
    f"{len(controller.valid_predict_params)} threshold(s) found that jointly "
    f"guarantee a precision >= {target_precision} and a recall >= {target_recall} "
    f"for every class, with 90% confidence."
)
print(f"Best threshold (secondary objective: recall of class {class_labels[0]}): "
      f"{controller.best_predict_param}")

9 threshold(s) found that jointly guarantee a precision >= 0.7 and a recall >= 0.7 for every class, with 90% confidence.
Best threshold (secondary objective: recall of class 0): 0.4289473684210526


/Users/trezel/Documents/code/MAPIE/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/trezel/Documents/code/MAPIE/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/trezel/Documents/code/MAPIE/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul



#### Visualizing precision and recall by threshold, per class

For each class, we plot its one-vs-rest precision and recall as a function of
the candidate thresholds, highlighting which thresholds are jointly valid
(green background) and the best threshold selected by the controller.

In [20]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

tested_thresholds = controller._predict_params
valid_thresholds = set(controller.valid_predict_params.tolist())
is_valid = np.array([t in valid_thresholds for t in tested_thresholds])

predictions_per_param = controller._get_predictions_per_param(
    X, tested_thresholds
)

fig = make_subplots(
    rows=1,
    cols=n_classes,
    subplot_titles=[f"Class {class_label}" for class_label in class_labels],
    shared_yaxes=True,
)

for i, (class_label, precision_risk, recall_risk) in enumerate(
    zip(class_labels, precision_risks, recall_risks)
):
    col = i + 1
    precision_values = np.array(
        [
            1 - precision_risk.get_value_and_effective_sample_size(y, y_pred)[0]
            for y_pred in predictions_per_param
        ]
    )
    recall_values = np.array(
        [
            1 - recall_risk.get_value_and_effective_sample_size(y, y_pred)[0]
            for y_pred in predictions_per_param
        ]
    )

    # add_vrect must come after at least one trace exists on this subplot,
    # otherwise plotly can't resolve the row/col axis reference and silently
    # drops the shape.
    fig.add_trace(
        go.Scatter(
            x=tested_thresholds,
            y=precision_values,
            name="Precision",
            legendgroup="precision",
            showlegend=(i == 0),
            line=dict(color="#1f77b4"),
        ),
        row=1,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=tested_thresholds,
            y=recall_values,
            name="Recall",
            legendgroup="recall",
            showlegend=(i == 0),
            line=dict(color="#ff7f0e"),
        ),
        row=1,
        col=col,
    )

    if is_valid.any():
        fig.add_vrect(
            x0=tested_thresholds[is_valid].min(),
            x1=tested_thresholds[is_valid].max(),
            fillcolor="green",
            opacity=0.15,
            line_width=0,
            layer="below",
            row=1,
            col=col,
        )
    fig.add_hline(
        y=target_precision, line_dash="dash", line_color="#1f77b4", opacity=0.6,
        row=1, col=col,
    )
    fig.add_hline(
        y=target_recall, line_dash="dash", line_color="#ff7f0e", opacity=0.6,
        row=1, col=col,
    )
    if controller.best_predict_param is not None:
        fig.add_vline(
            x=controller.best_predict_param,
            line_dash="dot",
            line_color="black",
            row=1,
            col=col,
        )
    fig.update_xaxes(title_text="Threshold", row=1, col=col)

fig.update_yaxes(title_text="Metric value", row=1, col=1)
fig.update_layout(
    title_text="One-vs-rest precision and recall by threshold",
    width=350 * n_classes,
    height=450,
)
fig.show()

/Users/trezel/Documents/code/MAPIE/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

divide by zero encountered in matmul

/Users/trezel/Documents/code/MAPIE/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

overflow encountered in matmul

/Users/trezel/Documents/code/MAPIE/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning:

invalid value encountered in matmul

